# Notebook 02: Pre-training Payments Foundation Model on PaySim

This notebook performs self-supervised Masked Event Prediction pre-training on the PaySim transaction corpus using the PRAGMA dual-encoder backbone. The output checkpoint serves as the foundation model for downstream adapter fine-tuning.

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Finai-research'
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
else:
    PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

import config
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from pragma_model import PRAGMA
from data.paysim_dataset import PaySimDataset

paths = config.setup_environment()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Initialize PaySim PRAGMA Model

In [ ]:
profile_config = config.PRAGMA_PAYSIM_PROFILE_CONFIG
event_config = config.PRAGMA_PAYSIM_EVENT_CONFIG

model = PRAGMA(profile_config, event_config, embed_dim=64)
model.to(device)
print('PRAGMA model initialized for PaySim pretraining.')

## 2. Pre-training Loop (Masked Event Prediction)

In [ ]:
def pretrain_epoch(model, dataloader, optimizer, criterion, mask_prob=0.15):
    model.train()
    total_loss = 0.0
    for batch_idx, batch in enumerate(dataloader):
        x_num, x_cat, events, seq_lengths, _ = [b.to(device) for b in batch]
        
        labels = events.clone()
        mask = torch.rand(events.shape[:2], device=events.device) < mask_prob
        pad_mask = torch.arange(events.shape[1], device=events.device)[None, :] >= seq_lengths[:, None]
        mask = mask & ~pad_mask
        
        masked_events = events.clone()
        masked_events[mask] = 0.0
        
        optimizer.zero_grad()
        fused, mlm_preds = model(x_num, x_cat, masked_events, seq_lengths, pretrain=True)
        
        if mask.sum() > 0:
            loss = criterion(mlm_preds[mask], labels[mask])
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        if batch_idx % 20 == 0:
            print(f'Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f}')
            
    return total_loss / max(1, len(dataloader))

paysim_train_path = paths['data_processed'] / 'paysim_train.parquet'
if paysim_train_path.exists():
    dataset = PaySimDataset(paysim_train_path, max_seq_len=200)
    loader = DataLoader(dataset, batch_size=64, shuffle=True)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    
    epochs = 3
    print(f'Starting pre-training for {epochs} epochs...')
    for ep in range(epochs):
        loss = pretrain_epoch(model, loader, optimizer, criterion)
        print(f'Epoch {ep+1}/{epochs} Pretraining Loss: {loss:.4f}')
        
    save_path = paths['models'] / 'pragma_pretrained_paysim.pth'
    torch.save(model.state_dict(), save_path)
    print(f'Saved foundation model checkpoint to {save_path}')
else:
    print(f'File {paysim_train_path} not found. Run Notebook 01 first.')